# Data Mining Lab 3.1–3.6: FPT Shop Product Dataset Analysis

**Workflow:** Understand → Describe → Visualize → Compare → Prepare for Classification → Generalize & Summarize

**Dataset:** Products crawled from FPT Shop (fptshop.com.vn)  
**Source file:** `fptshop_clean.csv`  
**Number of records:** 1,100 products across 8 categories


## 1. Dataset Overview and Preprocessing

### 1.1 Load data and inspect structure


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

df = pd.read_csv('/home/workdir/attachments/fptshop_clean.csv')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
df.head()


### 1.2 Cleaning steps
- Brand name normalization (OPPO/Oppo, ASUS/Asus, etc.)
- Fill missing `discount` from (price_original − price_sale) / price_original when possible; otherwise 0
- Fill missing `price_original` with `price_sale`
- Create English category labels and price-in-million columns for readability


In [ ]:
def clean_brand(b):
    if pd.isna(b): return 'Unknown'
    b = str(b).strip()
    mapping = {
        'Oppo': 'OPPO', 'ASUS': 'Asus', 'AQUA': 'Aqua', 'TECNO': 'Tecno',
        'Viewsonic': 'ViewSonic', 'Edra': 'EDRA', 'RedMagic': 'REDMAGIC',
        'cong': 'Unknown', 'di': 'Unknown', 'Đồ': 'Unknown', 'đồ': 'Unknown',
        'Di': 'Unknown', 'Cong': 'Unknown', 'Màn': 'Unknown', 'sấy': 'Unknown',
        'Tháp': 'Unknown', 'Black': 'Unknown', 'cửa': 'Unknown', 'Gaming': 'Gaming'
    }
    return mapping.get(b, b)

df['brand_clean'] = df['brand'].apply(clean_brand)

df['discount_filled'] = df['discount']
mask = df['discount'].isna() & df['price_original'].notna() & (df['price_original'] > 0)
df.loc[mask, 'discount_filled'] = (
    (df.loc[mask, 'price_original'] - df.loc[mask, 'price_sale']) / df.loc[mask, 'price_original'] * 100
).round(1)
df['discount_filled'] = df['discount_filled'].fillna(0).clip(0, 100)

df['price_original_filled'] = df['price_original'].fillna(df['price_sale'])
df['price_sale_m'] = df['price_sale'] / 1e6
df['price_original_m'] = df['price_original_filled'] / 1e6

cat_map = {
    'dien_thoai': 'Smartphone', 'laptop': 'Laptop', 'tu_lanh': 'Refrigerator',
    'may_lanh': 'Air Conditioner', 'may_giat': 'Washing Machine',
    'tivi': 'TV', 'man_hinh': 'Monitor', 'may_tinh_bang': 'Tablet'
}
df['category_en'] = df['category'].map(cat_map)

print('After cleaning – missing:')
print(df[['price_sale', 'price_original_filled', 'discount_filled']].isnull().sum())
print('\nCategory distribution:')
print(df['category_en'].value_counts())


## 2. Section 3.1 – Data Classification

**Objects:** Each row is one product listing (1,100 objects).  
**Attributes:** 8 original + derived attributes.

### Attribute type classification (at least 8 attributes)

| Attribute | Type | Reason | Possible use |
|-----------|------|--------|--------------|
| category / category_en | Nominal | Unordered product categories | Grouping, stratification, information gain |
| brand / brand_clean | Nominal | Brand names have no inherent order | Market share, brand-price analysis |
| name | Nominal (textual) | Unique product identifiers | Labeling, string similarity (optional) |
| url | Nominal | Unique URL strings | Key / reference |
| is_contact | Nominal (binary) | True/False flag (all False in this crawl) | Filter |
| price_sale | Ratio | Absolute zero, meaningful differences & ratios | Core numeric feature, pricing models |
| price_original | Ratio | Same as price_sale | Discount calculation, original value |
| discount | Ratio | Percentage, absolute zero, ratios meaningful | Promotion analysis, customer value |
| price_class (derived) | Ordinal | Low < Medium < High ordered levels | Classification target |
| discount_level (derived) | Ordinal | Low < Medium < High | Feature for classification |

**Overall structure:** Mixed numerical (ratio) and categorical (nominal) attributes typical of e-commerce catalog data. Strong positive skew in prices; moderate missingness in original price and discount that was imputed.


## 3. Section 3.2 – Descriptive Statistics

Selected numerical attributes: `price_sale`, `price_original_filled`, `discount_filled`.


In [ ]:
num_attrs = ['price_sale', 'price_original_filled', 'discount_filled']
rows = []
for col in num_attrs:
    s = df[col]
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75])
    rows.append({
        'Attribute': col,
        'Count': len(s),
        'Mean': round(s.mean(), 2),
        'Median': round(s.median(), 2),
        'Std': round(s.std(), 2),
        'Min': round(s.min(), 2),
        'Max': round(s.max(), 2),
        'Range': round(s.max() - s.min(), 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(q3 - q1, 2),
        'Skewness': round(s.skew(), 2)
    })
stats_table = pd.DataFrame(rows)
stats_table


### Interpretation
- **price_sale**: Mean ≈ 20.8M VND, Median ≈ 13.0M VND → strong right skew (skewness ≈ 4.2). Many affordable products and a long tail of premium/luxury items (max ≈ 297M). High variability (std ≈ 23.6M, IQR ≈ 22M).
- **price_original_filled**: Similar pattern, slightly higher mean/median as expected.
- **discount_filled**: Mean ≈ 15%, Median ≈ 11%. Less skewed. Most discounts between 0–25%; max 50%.
- Attributes with high variability / potential outliers: price_sale and price_original (luxury flagship phones, high-end laptops, large TVs/refrigerators).
- Business insight: FPT Shop catalog is dominated by mid-range electronics with occasional deep discounts; price distribution is typical of a multi-category retailer.


## 4. Section 3.3 – Data Visualization


In [ ]:
# 1. Histogram of sale price
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['price_sale_m'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Sale Price (Million VND)')
ax.set_ylabel('Frequency')
ax.set_title('Histogram of Product Sale Price')
ax.axvline(df['price_sale_m'].mean(), color='red', linestyle='--', label=f'Mean = {df["price_sale_m"].mean():.1f}M')
ax.axvline(df['price_sale_m'].median(), color='orange', linestyle='--', label=f'Median = {df["price_sale_m"].median():.1f}M')
ax.legend()
plt.tight_layout()
plt.show()
print('Interpretation: Right-skewed distribution; majority of products priced under 20M VND.')


In [ ]:
# 2. Boxplot by category
fig, ax = plt.subplots(figsize=(12, 6))
order = df.groupby('category_en')['price_sale_m'].median().sort_values().index
sns.boxplot(data=df, x='category_en', y='price_sale_m', order=order, ax=ax, palette='Set2')
ax.set_xlabel('Category')
ax.set_ylabel('Sale Price (Million VND)')
ax.set_title('Boxplot of Sale Price by Product Category')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
print('Interpretation: Laptops and TVs show highest median prices and many high-end outliers; Monitors and budget phones are cheapest.')


In [ ]:
# 3. Quantile plot
fig, ax = plt.subplots(figsize=(8, 5))
sorted_prices = np.sort(df['price_sale_m'].values)
n = len(sorted_prices)
quantiles = np.arange(1, n+1) / (n+1)
ax.plot(quantiles, sorted_prices, 'b-', linewidth=1.5)
ax.set_xlabel('Empirical Quantile')
ax.set_ylabel('Sale Price (Million VND)')
ax.set_title('Quantile Plot of Sale Price')
plt.tight_layout()
plt.show()
print('Interpretation: Steep rise in the upper quantiles confirms heavy right tail (luxury products).')


In [ ]:
# 4. Scatter price vs discount
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(df['price_sale_m'], df['discount_filled'], c=df['discount_filled'],
                cmap='viridis', alpha=0.5, s=20)
ax.set_xlabel('Sale Price (Million VND)')
ax.set_ylabel('Discount (%)')
ax.set_title('Scatter Plot: Sale Price vs Discount Percentage')
plt.colorbar(sc, ax=ax, label='Discount %')
plt.tight_layout()
plt.show()
print('Interpretation: Higher-priced items tend to have lower discounts (negative correlation); deep discounts (>30%) concentrate on mid/low-price products.')


In [ ]:
# 5. Multi-attribute visualization
fig, ax1 = plt.subplots(figsize=(11, 6))
cat_summary = df.groupby('category_en').agg(
    count=('name', 'count'),
    avg_price=('price_sale_m', 'mean')
).sort_values('count', ascending=False)
colors = plt.cm.viridis(cat_summary['avg_price'] / cat_summary['avg_price'].max())
bars = ax1.bar(cat_summary.index, cat_summary['count'], color=colors, edgecolor='black', alpha=0.85)
ax1.set_ylabel('Number of Products')
ax1.set_xlabel('Category')
ax1.set_title('Product Count by Category (bar color encodes average price)')
ax1.tick_params(axis='x', rotation=30)
for bar, (idx, row) in zip(bars, cat_summary.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f'{row["avg_price"]:.1f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('Interpretation: Laptop is the largest category and also the most expensive on average; Monitor has many listings but low average price.')


## 5. Section 3.4 – Measuring Similarities and Differences

We select 12 smartphones and two ratio attributes (`price_sale`, `discount_filled`).  
Attributes are **Min-Max normalized** to [0,1] because they have different scales.  
**Euclidean distance** is used (appropriate for continuous normalized features; treats both dimensions equally after scaling).


In [ ]:
phones = df[df['category'] == 'dien_thoai'].sample(12, random_state=42).reset_index(drop=True)
print(phones[['name', 'price_sale', 'discount_filled', 'brand_clean']].to_string())

attrs = ['price_sale', 'discount_filled']
X = phones[attrs].values
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X)

dist_eucl = pairwise_distances(X_norm, metric='euclidean')
dist_df = pd.DataFrame(dist_eucl,
                       index=phones['name'].str[:35],
                       columns=phones['name'].str[:35])
print('\nEuclidean distance matrix:')
display(dist_df.round(3))

n = len(phones)
pairs = [(i, j, dist_eucl[i, j]) for i in range(n) for j in range(i+1, n)]
pairs_sorted = sorted(pairs, key=lambda x: x[2])
print('\n=== 3 most similar pairs ===')
for i, j, d in pairs_sorted[:3]:
    print(f'{phones.loc[i,"name"][:40]}  <->  {phones.loc[j,"name"][:40]}  | d={d:.4f}')
print('\n=== 3 most different pairs ===')
for i, j, d in pairs_sorted[-3:][::-1]:
    print(f'{phones.loc[i,"name"][:40]}  <->  {phones.loc[j,"name"][:40]}  | d={d:.4f}')


### Interpretation
The numerical distances align with business intuition: low-cost feature phones cluster together (small distance), while flagship foldables / high-end iPhones are far from budget models. Distance on (price, discount) therefore captures meaningful product similarity for pricing/promotion analysis.


## 6. Section 3.5 – Measuring Data for Classification

### 6.1 Discretization – target variable `price_class`
Intervals chosen by **empirical tertiles** (≈ equal frequency) so that classes are balanced:
- Low: price ≤ 33rd percentile
- Medium: 33rd–66th percentile
- High: > 66th percentile


In [ ]:
q33, q66 = df['price_sale'].quantile([0.33, 0.66])
def price_class(p):
    if p <= q33: return 'Low'
    elif p <= q66: return 'Medium'
    else: return 'High'
df['price_class'] = df['price_sale'].apply(price_class)
print(df['price_class'].value_counts())
print(f'Thresholds: Low ≤ {q33:,.0f} VND | Medium ≤ {q66:,.0f} VND | High > {q66:,.0f} VND')


In [ ]:
def entropy(series):
    counts = series.value_counts(normalize=True)
    return -np.sum(counts * np.log2(counts + 1e-12))

H = entropy(df['price_class'])
print(f'Entropy of price_class: {H:.4f} bits (max for 3 classes = log2(3) ≈ 1.585)')


In [ ]:
def info_gain(df, attr, target='price_class'):
    H_parent = entropy(df[target])
    total = len(df)
    H_cond = sum((len(g)/total) * entropy(g[target]) for _, g in df.groupby(attr))
    return H_parent - H_cond

top_brands = df['brand_clean'].value_counts().head(15).index
df['brand_top'] = df['brand_clean'].where(df['brand_clean'].isin(top_brands), 'Other')
df['discount_level'] = pd.cut(df['discount_filled'], bins=[-0.1, 10, 25, 100],
                              labels=['Low', 'Medium', 'High'])

ig = {
    'category_en': info_gain(df, 'category_en'),
    'brand_top': info_gain(df, 'brand_top'),
    'discount_level': info_gain(df, 'discount_level')
}
ig_df = pd.DataFrame(list(ig.items()), columns=['Attribute', 'Information Gain']).sort_values('Information Gain', ascending=False)
print(ig_df.to_string(index=False))
print('\nRanking: category > brand > discount_level')


In [ ]:
# Chi-square tests
ct1 = pd.crosstab(df['price_class'], df['category_en'])
chi2, p, dof, _ = stats.chi2_contingency(ct1)
print(f'Chi-square (price_class × category): χ² = {chi2:.2f}, p-value = {p:.2e}, dof = {dof}')
print('→ Strong association (p << 0.05): category is highly informative for price class.\n')

ct2 = pd.crosstab(df['price_class'], df['brand_top'])
chi2b, pb, dofb, _ = stats.chi2_contingency(ct2)
print(f'Chi-square (price_class × brand_top): χ² = {chi2b:.2f}, p-value = {pb:.2e}, dof = {dofb}')
print('→ Strong association: brand also related to price class.')


In [ ]:
# Correlations
c1 = df['price_sale'].corr(df['discount_filled'])
c2 = df['price_sale'].corr(df['price_original_filled'])
print(f'Pearson corr(price_sale, discount_filled) = {c1:.4f}  → moderate negative')
print(f'Pearson corr(price_sale, price_original)  = {c2:.4f}  → very strong positive')
print('Note: correlation ≠ causation. High original price naturally co-occurs with high sale price;')
print('deep discounts are more common on lower-priced items (clearance / volume strategy).')


## 7. Section 3.6 – Data Generalization and Summarization

### Concept hierarchies
1. **Product → Category** (already present: dien_thoai → Smartphone, …)
2. **Exact price → Price level**  
   Budget (<5M) → Mid-range (5–15M) → Premium (15–30M) → Luxury (>30M)
3. **Discount → Discount level** (Low / Medium / High)


In [ ]:
def price_level(p):
    if p < 5e6: return 'Budget (<5M)'
    elif p < 15e6: return 'Mid-range (5-15M)'
    elif p < 30e6: return 'Premium (15-30M)'
    else: return 'Luxury (>30M)'
df['price_level'] = df['price_sale'].apply(price_level)

cube = df.groupby(['category_en', 'price_level']).agg(
    product_count=('name', 'count'),
    avg_price_m=('price_sale_m', 'mean'),
    avg_discount=('discount_filled', 'mean')
).reset_index()
print('Data cube (category × price_level):')
display(cube)


In [ ]:
summary = df.groupby('category_en').agg(
    n_products=('name', 'count'),
    avg_price_m=('price_sale_m', 'mean'),
    median_price_m=('price_sale_m', 'median'),
    avg_discount=('discount_filled', 'mean'),
    top_brand=('brand_clean', lambda x: x.value_counts().index[0])
).sort_values('n_products', ascending=False)
print('Higher-level category summary:')
display(summary.round(2))


### Information lost vs. analytical benefits
- **Lost:** Individual product names, exact prices, specific model differences, URL-level identity.
- **Gained:** Compact view of the catalog structure, easy comparison of average price/discount across categories and price tiers, readiness for OLAP-style queries and management dashboards. The cube supports roll-up (all categories) and drill-down (specific category × price level).


## 8. Overall Findings and Key Insights

1. The FPT Shop dataset (1,100 products) is a typical multi-category e-commerce catalog with heavy right-skewed prices.
2. Laptops dominate both in quantity and average price; Monitors are numerous but inexpensive.
3. Category is the strongest predictor of price class (highest information gain + highly significant χ²).
4. Discount is moderately negatively correlated with price – premium items are less heavily discounted.
5. Euclidean distance on normalized (price, discount) successfully recovers intuitive product similarity among smartphones.
6. Concept hierarchies and the data cube provide a clean, management-friendly summary while sacrificing fine-grained product identity.

**Recommendation for further work:** Add ratings / review counts if available, engineer brand-tier features, and build a simple decision-tree classifier using the prepared `price_class` target.


## Appendix – Reproducibility
- Source CSV: `/home/workdir/attachments/fptshop_clean.csv`
- All analysis performed with Python 3 (pandas, numpy, scipy, scikit-learn, matplotlib, seaborn).
- Random seed for phone sampling: 42.
